## 对比学习增强的 Transformer 异常检测

说明：本 notebook 复用 `baseline_transformer.ipynb` 中的数据预处理和 DataLoader，只替换模型与训练为“重构 + 对比学习联合损失”，然后按与基线相同的方式计算时间点级异常分数与指标（含 point-adjust）。

In [ ]:
from pathlib import Path
import os
import os
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from sklearn.preprocessing import StandardScaler

train_path = str(Path(os.environ.get("WADI_DATA_ROOT", "../data/WADI")) / "WADI_14days_new.csv")
test_path  = str(Path(os.environ.get("WADI_DATA_ROOT", "../data/WADI")) / "WADI_attackdataLABLE.csv")

assert os.path.exists(train_path), f"not found: {train_path}"
assert os.path.exists(test_path), f"not found: {test_path}"

train_df = pd.read_csv(train_path)
test_raw = pd.read_csv(test_path)

print("train_df:", train_df.shape)
print("test_raw:", test_raw.shape)

# 处理测试集表头与标签
test_raw.columns = test_raw.iloc[0]
test_raw = test_raw.iloc[1:].reset_index(drop=True)
if len(test_raw) >= 2:
    test_raw = test_raw.iloc[:-2]

label_col = "Attack LABLE (1:No Attack, -1:Attack)"
assert label_col in test_raw.columns, f"label column not found: {label_col}"

test_labels = test_raw[label_col].replace({1: 0, -1: 1}).astype(int).to_numpy()
test_df = test_raw.drop(columns=[label_col])

# drop meta
for c in ["Row ", "Date ", "Time"]:
    if c in test_df.columns:
        test_df = test_df.drop(columns=[c])
for c in ["Row", "Date", "Time"]:
    if c in train_df.columns:
        train_df = train_df.drop(columns=[c])

# 数值化
train_df = train_df.dropna(how="all").astype(float)
test_df = test_df.astype(float)

# NaN 清洗：删除全为 NaN 的列，其余用训练集中位数填充
all_nan_cols_train = train_df.columns[train_df.isna().all(axis=0)]
all_nan_cols_test  = test_df.columns[test_df.isna().all(axis=0)]
all_nan_cols = sorted(list(set(all_nan_cols_train).union(set(all_nan_cols_test))))

if len(all_nan_cols) > 0:
    print("Drop all-NaN columns:", len(all_nan_cols))
    train_df = train_df.drop(columns=all_nan_cols)
    test_df  = test_df.drop(columns=all_nan_cols)

train_median = train_df.median()
train_df = train_df.fillna(train_median)
test_df  = test_df.fillna(train_median)

print("train_df clean:", train_df.shape)
print("test_df  clean:", test_df.shape)
print("test_labels:", test_labels.shape, "anomaly count:", int(test_labels.sum()))

# 标准化（只用训练集）
scaler = StandardScaler()
train_data = scaler.fit_transform(train_df.values)
test_data  = scaler.transform(test_df.values)

train_data = np.nan_to_num(train_data, nan=0.0, posinf=0.0, neginf=0.0)
test_data  = np.nan_to_num(test_data,  nan=0.0, posinf=0.0, neginf=0.0)

D = train_data.shape[1]
print("feature dim D =", D)

# 构建 Dataset/DataLoader（与基线一致）
from torch.utils.data import Dataset, DataLoader

class SlidingWindowWithIndexDataset(Dataset):
    def __init__(self, data: np.ndarray, window_size: int, stride: int = 1):
        assert stride >= 1
        self.data = torch.tensor(data, dtype=torch.float32)
        self.window_size = window_size
        self.stride = stride

    def __len__(self):
        max_start = len(self.data) - self.window_size
        if max_start < 0:
            return 0
        return max_start // self.stride + 1

    def __getitem__(self, i: int):
        start = i * self.stride
        x = self.data[start : start + self.window_size]
        return x, start

window_size = 100
batch_size = 64
train_stride = 5

train_ds_train = SlidingWindowWithIndexDataset(train_data, window_size, stride=train_stride)
train_ds_eval  = SlidingWindowWithIndexDataset(train_data, window_size, stride=1)
test_ds        = SlidingWindowWithIndexDataset(test_data,  window_size, stride=1)

train_loader = DataLoader(train_ds_train, batch_size=batch_size, shuffle=True, drop_last=False)
train_eval_loader = DataLoader(train_ds_eval, batch_size=batch_size, shuffle=False, drop_last=False)
test_loader = DataLoader(test_ds, batch_size=batch_size, shuffle=False, drop_last=False)

xb, ib = next(iter(train_loader))
print("batch x:", xb.shape, "batch idx:", ib.shape)

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
from sklearn.metrics import precision_recall_fscore_support

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

In [ ]:
# ============ 时间序列增强（对比学习用） ============

def jitter(x: torch.Tensor, sigma: float = 0.01) -> torch.Tensor:
    if sigma <= 0:
        return x
    return x + torch.randn_like(x) * sigma


def scaling(x: torch.Tensor, sigma: float = 0.1) -> torch.Tensor:
    if sigma <= 0:
        return x
    B, L, D = x.shape
    scale = torch.randn(B, 1, D, device=x.device) * sigma + 1.0
    return x * scale


def time_mask(x: torch.Tensor, mask_ratio: float = 0.05) -> torch.Tensor:
    if mask_ratio <= 0:
        return x
    B, L, D = x.shape
    num_mask = int(L * mask_ratio)
    if num_mask <= 0:
        return x
    x_masked = x.clone()
    for b in range(B):
        idx = torch.randperm(L, device=x.device)[:num_mask]
        x_masked[b, idx, :] = 0.0
    return x_masked


def feature_mask(x: torch.Tensor, mask_ratio: float = 0.05) -> torch.Tensor:
    if mask_ratio <= 0:
        return x
    B, L, D = x.shape
    num_mask = int(D * mask_ratio)
    if num_mask <= 0:
        return x
    x_masked = x.clone()
    for b in range(B):
        idx = torch.randperm(D, device=x.device)[:num_mask]
        x_masked[b, :, idx] = 0.0
    return x_masked


def augment_ts(x: torch.Tensor) -> torch.Tensor:
    x1 = jitter(x, sigma=0.01)
    x1 = scaling(x1, sigma=0.1)
    x1 = time_mask(x1, mask_ratio=0.05)
    x1 = feature_mask(x1, mask_ratio=0.05)
    return x1

In [ ]:
# ============ 带对比学习模块的 Transformer ============

class PositionalEncoding(nn.Module):
    def __init__(self, d_model: int, max_len: int = 2048, dropout: float = 0.1):
        super().__init__()
        self.dropout = nn.Dropout(dropout)
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float32).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2, dtype=torch.float32) * (-np.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer("pe", pe.unsqueeze(0))

    def forward(self, x: torch.Tensor):
        return self.dropout(x + self.pe[:, : x.size(1)])


class ContrastiveTransformer(nn.Module):
    def __init__(self, input_dim: int, d_model: int = 256, nhead: int = 8,
                 num_encoder_layers: int = 3, num_decoder_layers: int = 3,
                 dim_feedforward: int = 512, dropout: float = 0.1,
                 proj_dim: int = 128):
        super().__init__()
        self.input_proj = nn.Linear(input_dim, d_model)
        self.pos_enc = PositionalEncoding(d_model=d_model, dropout=dropout)

        enc_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead, dim_feedforward=dim_feedforward,
            dropout=dropout, batch_first=True, activation="gelu", norm_first=True
        )
        dec_layer = nn.TransformerDecoderLayer(
            d_model=d_model, nhead=nhead, dim_feedforward=dim_feedforward,
            dropout=dropout, batch_first=True, activation="gelu", norm_first=True
        )
        self.encoder = nn.TransformerEncoder(enc_layer, num_layers=num_encoder_layers)
        self.decoder = nn.TransformerDecoder(dec_layer, num_layers=num_decoder_layers)
        self.output_proj = nn.Linear(d_model, input_dim)

        self.proj_head = nn.Sequential(
            nn.Linear(d_model, d_model),
            nn.ReLU(inplace=True),
            nn.Linear(d_model, proj_dim),
        )

    def encode(self, x: torch.Tensor):
        x_emb = self.pos_enc(self.input_proj(x))
        h = self.encoder(x_emb)          # [B,L,d_model]
        h_pool = h.mean(dim=1)           # [B,d_model]
        z = self.proj_head(h_pool)       # [B,proj_dim]
        z = F.normalize(z, dim=1)
        return h, z

    def forward(self, x: torch.Tensor):
        h, _ = self.encode(x)
        tgt_emb = self.pos_enc(self.input_proj(x))
        dec_out = self.decoder(tgt_emb, h)
        return self.output_proj(dec_out)


# D 由 baseline 中定义
model = ContrastiveTransformer(input_dim=D).to(device)

with torch.no_grad():
    x0, _ = next(iter(train_loader))
    x0 = x0.to(device)
    y0 = model(x0)
    print("x0:", x0.shape, "y0:", y0.shape)

In [ ]:
# ============ InfoNCE 对比损失 ============

def info_nce_loss(z1: torch.Tensor, z2: torch.Tensor, temperature: float = 0.1) -> torch.Tensor:
    """z1, z2: [B, d]，同一行是正对，batch 内其余为负样本"""
    B, _ = z1.shape
    z1 = F.normalize(z1, dim=1)
    z2 = F.normalize(z2, dim=1)

    logits = (z1 @ z2.t()) / temperature  # [B,B]
    labels = torch.arange(B, device=z1.device, dtype=torch.long)
    loss1 = F.cross_entropy(logits, labels)
    loss2 = F.cross_entropy(logits.t(), labels)
    return 0.5 * (loss1 + loss2)

In [ ]:
# ============ 联合训练：重构 + 对比学习 ============

lr = 1e-4
weight_decay = 1e-4
epochs = 20
lambda_cl = 0.1
temperature = 0.2

optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
recon_criterion = nn.MSELoss(reduction="mean")


def train_one_epoch(model, loader):
    model.train()
    total_recon = 0.0
    total_cl = 0.0
    n = 0
    for x, _idx in loader:
        x = x.to(device)

        # 两个增强视图
        x1 = augment_ts(x)
        x2 = augment_ts(x)

        optimizer.zero_grad(set_to_none=True)

        # 重构损失（用原始 x）
        x_hat = model(x)
        loss_recon = recon_criterion(x_hat, x)

        # 对比损失（用增强视图）
        _, z1 = model.encode(x1)
        _, z2 = model.encode(x2)
        loss_cl = info_nce_loss(z1, z2, temperature=temperature)

        loss = loss_recon + lambda_cl * loss_cl
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        bs = x.size(0)
        total_recon += float(loss_recon.item()) * bs
        total_cl += float(loss_cl.item()) * bs
        n += bs

    return total_recon / max(n, 1), total_cl / max(n, 1)


for ep in range(1, epochs + 1):
    recon_loss, cl_loss = train_one_epoch(model, train_loader)
    scheduler.step()
    total_loss = recon_loss + lambda_cl * cl_loss
    print(f"Epoch {ep:03d} | recon={recon_loss:.6f} | cl={cl_loss:.6f} | total={total_loss:.6f} | lr={scheduler.get_last_lr()[0]:.2e}")

In [ ]:
# ============ 用对比学习模型重新计算时间点级分数 ============

@torch.no_grad()
def compute_point_scores(model, loader, seq_len: int):
    model.eval()
    score_sum = np.zeros(seq_len, dtype=np.float64)
    count = np.zeros(seq_len, dtype=np.int64)

    for x, idx in loader:
        x = x.to(device)
        idx = idx.numpy()
        x_hat = model(x)
        err = (x_hat - x).pow(2).mean(dim=2).detach().cpu().numpy()  # [B,L]
        B, L = err.shape
        for b in range(B):
            s = int(idx[b])
            e = s + L
            score_sum[s:e] += err[b]
            count[s:e] += 1

    return score_sum / np.maximum(count, 1)

train_point_scores = compute_point_scores(model, train_eval_loader, seq_len=len(train_data))
test_point_scores  = compute_point_scores(model, test_loader,      seq_len=len(test_data))

print("train_point_scores:", train_point_scores.shape)
print("test_point_scores :", test_point_scores.shape)

In [ ]:
# ============ 阈值 + 点级指标（与基线保持同样 q） ============

quantile = 0.95
threshold = float(np.quantile(train_point_scores, quantile))

pred = (test_point_scores > threshold).astype(int)

p, r, f1, _ = precision_recall_fscore_support(test_labels, pred, average="binary", zero_division=0)
print(f"threshold@q={quantile}: {threshold:.6f}")
print(f"Point-wise P={p:.4f} R={r:.4f} F1={f1:.4f}")

In [ ]:
# ============ point-adjust（事件级）指标 ============

def prf1(y_true, y_pred):
    p, r, f1, _ = precision_recall_fscore_support(
        y_true, y_pred, average="binary", zero_division=0
    )
    return float(p), float(r), float(f1)


def point_adjust(y_true: np.ndarray, y_pred: np.ndarray) -> np.ndarray:
    y_true = y_true.astype(int)
    y_pred = y_pred.astype(int).copy()
    n = len(y_true)
    i = 0
    while i < n:
        if y_true[i] == 1:
            j = i
            while j < n and y_true[j] == 1:
                j += 1
            if y_pred[i:j].max() == 1:
                y_pred[i:j] = 1
            i = j
        else:
            i += 1
    return y_pred

raw_p, raw_r, raw_f1 = prf1(test_labels, pred)
pa_pred = point_adjust(test_labels, pred)
pa_p, pa_r, pa_f1 = prf1(test_labels, pa_pred)

print(f"Raw     P={raw_p:.4f} R={raw_r:.4f} F1={raw_f1:.4f}")
print(f"PA      P={pa_p:.4f} R={pa_r:.4f} F1={pa_f1:.4f}")